# FinalNet-ASPP-FPN-512 on the Figshare Brain Tumor Dataset — Experiment 2b (patient-level protocol, 512×512)

Controlled architectural experiment on top of `finalnet_figshare_experiment2.ipynb` (FinalNet, 256×256). Three changes only:
**(A)** ASPP at the EfficientNet-B0 bottleneck, **(B)** FPN-style top-down multi-scale fusion over the five encoder stages feeding the existing U-Net/scSE decoder, **(C)** 512×512 input.

Unchanged: pretrained EfficientNet-B0 encoder, scSE U-Net decoder, 3 deep-supervision heads + main head, soft-mask feature-fusion classifier (same input widths), 3-class labels,
Tversky(0.7)+BCE segmentation loss, CE (label smoothing 0.1), learned uncertainty weighting, AdamW (lr 2e-4, wd 1e-4), warm-up + cosine, EMA 0.999, grad-clip 1.0, AMP, 35 epochs, early stopping (patience 8),
effective batch 16 (now = micro-batch × gradient accumulation), 5-fold ensemble, all metric definitions, 0.5 threshold, patient-level split. No new loss, no TTA, no Grad-CAM.

**Dataset:** Figshare brain tumor dataset (Cheng et al.), Kaggle `ashkhagan/figshare-brain-tumor-dataset`.
3,064 T1-CE slices, 233 patients, `.mat` files (`cjdata.PID / image / label / tumorMask / tumorBorder`).
Label mapping (dataset README): **1 = meningioma, 2 = glioma, 3 = pituitary**. `cvind.mat` is *not* used.

**Flow:** MRI 512² → EfficientNet-B0 (strides 2/4/8/16/32) → **ASPP** on the 16×16 bottleneck (1×1 | 3×3 d=2,4,6 | global pool, residual, 320 ch) → **FPN** top-down fusion (stages 16/24/40/112/320 ch → 64 ch laterals, 3×3 smoothing, 1×1 adapters added back to each skip) → existing decoder (scSE) → main + 3 aux heads; classifier uses the ASPP bottleneck + the FPN-fused stride-16 skip weighted by the soft mask.

**Protocol (patient-level — no image-level splitting anywhere):**
1. Hold-out: ~20 % of *patients* (grouped by `PID`) are the final test set — **the same 629 images / 46 patients as the 256-px baseline** (the split cell reuses the baseline's `split_info.json` if attached, otherwise regenerates the identical deterministic split and asserts 629 / 46).
2. 5-fold stratified **group** CV on the remaining patients; a fold's validation part is used only for early stopping / checkpoint selection.
3. Five best-EMA fold checkpoints are ensembled (average class probabilities and mask probabilities, **then** threshold at 0.5).
4. The ensemble is evaluated **once** on the hold-out test set → Table A (classification) and Table B (segmentation).
5. Explicit assertions verify that no patient crosses train / validation / test.

**How to use on Kaggle (Internet may be OFF):** add the Figshare dataset via *Add Input*, GPU = T4, optionally attach the baseline run's output (`Figshare_Experiment_Results/split_info.json`) and an offline copy of `segmentation-models-pytorch`/EfficientNet-B0 weights if Internet is off (the notebook never pip-installs and never silently uses a randomly initialised encoder).
Run cells in order. Set `SMOKE_TEST_ONLY = True` to stop after the architecture smoke test (cell 8); `SMOKE = True` for a tiny end-to-end dry run.

All outputs go to `/kaggle/working/Figshare_Experiment_ASPP_FPN_512/`.

In [ ]:
# ---------------- imports (no automatic pip install: the notebook must also run with Kaggle Internet OFF) ----------------
import os, re, copy, math, time, json, random, inspect, glob, warnings
from collections import Counter
# core imports first, so later cells never hit NameError (nn / tqdm / F) even if an optional import below has a problem
import numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

# segmentation_models_pytorch is NOT preinstalled on Kaggle. Try to import; if missing, try a one-time pip install (needs Internet ON).
import subprocess, sys, importlib
def _need(mod):
    try: importlib.import_module(mod); return False
    except ImportError: return True
if _need("segmentation_models_pytorch") or _need("timm"):
    print("segmentation_models_pytorch / timm missing -> trying pip install (requires Kaggle Internet ON) ...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch", "timm"], capture_output=True, text=True)
    if r.returncode != 0:
        raise ImportError(
            "Could not install segmentation-models-pytorch (Internet is probably OFF).\n"
            "FIX: in the Kaggle right sidebar open 'Session options' -> turn 'Internet' ON (phone-verified account needed), then re-run this cell.\n"
            "Offline alternative: upload wheels (pip download segmentation-models-pytorch timm -d wheels/) as a Kaggle Dataset, attach it and run\n"
            "  !pip install --no-index --find-links=/kaggle/input/<your-wheel-dataset> segmentation-models-pytorch\n\n" + r.stderr[-800:])
    importlib.invalidate_caches()
import segmentation_models_pytorch as smp
import timm
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
import scipy, scipy.io as sio
from scipy import ndimage as ndi
try:
    import h5py                                   # needed only for MATLAB v7.3 (HDF5) .mat files
except ImportError:
    h5py = None
import sklearn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold          # group-aware stratification (patient-level)
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, roc_auc_score,
                             roc_curve, auc, confusion_matrix)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev, "| smp", smp.__version__, "| timm", timm.__version__, "| torch", torch.__version__,
      "| sklearn", sklearn.__version__, "| scipy", scipy.__version__, "| h5py", getattr(h5py, "__version__", None))
if dev.type == "cuda":
    _p = torch.cuda.get_device_properties(0); print(f"GPU: {_p.name} | {_p.total_memory / 2**30:.1f} GiB")

In [ ]:
# ---------------- CONFIG ----------------
DATASET = "FIGSHARE"   # Experiment 2: Figshare brain tumor dataset (Cheng et al.), T1-CE, 3 tumor classes
SMOKE   = False        # True = tiny quick test run (~30 patients, 2 epochs, 2 folds)
SMOKE_TEST_ONLY = False  # True = run everything up to and including the architecture smoke test (cell 8), then stop (no training)
AUTO_MICRO_BATCH = True  # cell 8 probes the largest micro-batch (a divisor of HP["batch"]) that fits this GPU and keeps the effective batch

CFG = dict(
    folder="figshare-brain-tumor-dataset",             # folder name of the Kaggle input (searched under /kaggle/input)
    size=512,                                          # FinalNet-ASPP-FPN-512: was 256 in finalnet_figshare_experiment2.ipynb
    class_weights=False,                               # unchanged default (original D1/D3 setting)
    classes=["meningioma", "glioma", "pituitary"],     # model class index = label - 1
    label_map={1: "meningioma", 2: "glioma", 3: "pituitary"},   # from the dataset README (cjdata.label)
    expected=dict(n=3064, patients=233,                # published dataset statistics (also verified by the user)
                  per_class=dict(meningioma=708, glioma=1426, pituitary=930),
                  patients_per_class=dict(meningioma=82, glioma=89, pituitary=62)),
    expected_test=dict(n_images=629, n_patients=46),   # hold-out test set of the 256-px baseline run (must be unchanged)
    reference_split_json=None,                         # optional path to the BASELINE run's split_info.json (see split cell); None = auto-search
    paper=None,                                        # no verified published Figshare result for this protocol (see paper-comparison cell)
)
STRICT_COUNTS = True   # raise if the loaded data does not match the published counts (3,064 / 233 / per-class)
STRICT_SPLIT  = True   # raise if the hold-out test set is not the baseline's 629 images / 46 patients

# Hyper-parameters: identical to the baseline notebook (epochs, patience, effective batch, lr, wd, ema, folds, test fraction, seed).
# NEW: "micro_batch" = images per forward/backward pass. Gradients are accumulated over batch/micro_batch micro-batches, so each
# optimiser step still sees the same 16 images (effective batch 16), as in the baseline. Only memory use changes.
HP = dict(epochs=35, patience=8, batch=16, lr=2e-4, wd=1e-4, ema=0.999,
          n_folds=5, test_frac=0.2, seed=42, workers=2, micro_batch=8)
if SMOKE: HP.update(epochs=2, n_folds=2, batch=8, micro_batch=4)
SPLIT_TRIES = 20 if SMOKE else 300     # number of seeded group-split candidates screened for class balance (labels/PIDs only)

def check_batch_cfg():
    assert HP["batch"] % HP["micro_batch"] == 0, f"effective batch {HP['batch']} must be a multiple of micro_batch {HP['micro_batch']}"
    HP["accum"] = HP["batch"] // HP["micro_batch"]
check_batch_cfg()
def eval_bs(): return HP["micro_batch"]      # validation / test batch size (no gradients -> same size is safely below the training peak)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(HP["seed"])
CLASSES = CFG["classes"]; NC = len(CLASSES); SIZE = CFG["size"]
assert SIZE == 512, "this notebook is the 512x512 experiment"
assert CLASSES == [CFG["label_map"][k] for k in sorted(CFG["label_map"])], "class order must follow label order 1,2,3"

OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
RES = f"{OUT}/Figshare_Experiment_ASPP_FPN_512"
for sub in ("", "checkpoints", "figures"): os.makedirs(f"{RES}/{sub}", exist_ok=True)
print("results dir:", RES, "| SIZE:", SIZE, "| seed:", HP["seed"],
      f"| effective batch {HP['batch']} = micro-batch {HP['micro_batch']} x accumulation {HP['accum']}")

def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return None if not np.isfinite(o) else float(o)
    if isinstance(o, float): return None if not math.isfinite(o) else o
    if isinstance(o, np.ndarray): return jsonable(o.tolist())
    return o

In [ ]:
# ---------------- locate dataset + show its layout (check this once!) ----------------
def find_root(folder):
    for dp, dn, _ in os.walk("/kaggle/input"):
        if os.path.basename(dp) == folder:
            return dp
    return "/kaggle/input"
ROOT = find_root(CFG["folder"])
print("ROOT:", ROOT)
for dp, dn, fs in os.walk(ROOT):
    depth = dp[len(ROOT):].count(os.sep)
    if depth <= 3:
        print("  " * depth + os.path.basename(dp) + f"/  ({len(fs)} files)  e.g. {fs[:2]}")

all_mat = sorted(glob.glob(f"{ROOT}/**/*.mat", recursive=True))
CVIND   = [f for f in all_mat if os.path.basename(f).lower() == "cvind.mat"]
MAT_FILES = [f for f in all_mat if os.path.basename(f).lower() != "cvind.mat"]
print(f"\n.mat files found: {len(all_mat)}  ->  sample files: {len(MAT_FILES)} | cvind.mat present: {bool(CVIND)} "
      f"(NOT used: we build our own patient-level folds)")
if not MAT_FILES:
    raise RuntimeError("No .mat sample files found under ROOT - check that the dataset was added via 'Add Input'.")

# show the label-related lines of any README shipped with the dataset (label mapping check)
docs = [f for ext in ("md", "txt") for f in glob.glob(f"{ROOT}/**/*.{ext}", recursive=True)]
for f in docs[:3]:
    try:
        txt = open(f, errors="ignore").read()
        hits = [l.strip() for l in txt.splitlines() if "label" in l.lower()][:6]
        print(f"\n[{os.path.relpath(f, ROOT)}] label-related lines:", hits if hits else "(none)")
    except Exception as e:
        print("could not read", f, e)
print("\nLabel mapping used (dataset README, cjdata.label):", CFG["label_map"])

In [ ]:
# ---------------- load (image, mask, label, PID) from the .mat files, resize to SIZE, verify ----------------
def read_cjdata(path):
    """Read one Figshare .mat file. MATLAB v5/v7 -> scipy.io.loadmat; MATLAB v7.3 (HDF5) -> h5py."""
    try:
        c = sio.loadmat(path, squeeze_me=True, struct_as_record=False)["cjdata"]
        img, msk = np.asarray(c.image), np.asarray(c.tumorMask)
        lab = np.asarray(c.label).reshape(-1)[0]
        pid = np.asarray(c.PID).reshape(-1)[0]
        fmt = "scipy"
    except NotImplementedError:                          # scipy refuses v7.3 files -> HDF5 reader
        if h5py is None: raise RuntimeError("MATLAB v7.3 file but h5py is not installed")
        with h5py.File(path, "r") as f:
            c = f["cjdata"]
            img, msk = np.array(c["image"]).T, np.array(c["tumorMask"]).T      # HDF5 stores MATLAB arrays transposed
            lab = np.array(c["label"]).reshape(-1)[0]
            pid = "".join(chr(int(v)) for v in np.array(c["PID"]).reshape(-1))  # char array stored as uint16 codes
        fmt = "h5py"
    return dict(image=img, mask=msk, label=int(lab), pid=str(pid).strip(), fmt=fmt)

def prep_image(im, size):
    """Per-slice min-max -> uint8 (the original pipeline expects 0..255 images), then resize to size x size."""
    im = im.astype(np.float32); lo, hi = float(im.min()), float(im.max())
    if hi <= lo: return None
    im8 = np.round((im - lo) / (hi - lo) * 255).astype(np.uint8)
    interp = cv2.INTER_AREA if im8.shape[0] > size else cv2.INTER_LINEAR      # 512-px slices pass through unchanged at size=512; 256-px slices are up-sampled bilinearly (AREA only if ever shrinking)
    return cv2.resize(im8, (size, size), interpolation=interp)

def prep_mask(m, size):
    """Binarise (>0) and resize with NEAREST (never bilinear/bicubic) so mask values stay strictly in {0,1}."""
    out = cv2.resize((np.asarray(m) > 0).astype(np.uint8), (size, size), interpolation=cv2.INTER_NEAREST)
    assert out.dtype == np.uint8 and int(out.max(initial=0)) <= 1
    return out

def natkey(p):
    s = os.path.splitext(os.path.basename(p))[0]
    return (0, int(s), p) if s.isdigit() else (1, 0, p)

Xl, Ml, rows, excluded = [], [], [], []
for p in tqdm(sorted(MAT_FILES, key=natkey), desc="reading .mat"):
    try:
        r = read_cjdata(p); im, mk = r["image"], r["mask"]
        if im.ndim != 2 or mk.ndim != 2:           raise ValueError(f"expected 2-D image/mask, got {im.shape}/{mk.shape}")
        if im.shape != mk.shape:                   raise ValueError(f"image/mask shape mismatch {im.shape} vs {mk.shape}")
        if r["label"] not in CFG["label_map"]:     raise ValueError(f"label {r['label']} not in {sorted(CFG['label_map'])}")
        if not r["pid"]:                           raise ValueError("empty PID")
        im_s = prep_image(im, SIZE)
        if im_s is None:                           raise ValueError("constant image")
        mk_s = prep_mask(mk, SIZE)
        H, W = im.shape; px0 = int((mk > 0).sum()); px1 = int(mk_s.sum())
        rows.append(dict(file=os.path.relpath(p, ROOT), pid=r["pid"], label=r["label"], H=H, W=W, fmt=r["fmt"],
                         img_dtype=str(im.dtype), mask_nonbinary=bool(set(np.unique(mk).tolist()) - {0, 1, True, False}),
                         mask_px_orig=px0, mask_px_resized=px1,
                         area_ratio=(px1 * (H * W) / (SIZE * SIZE)) / px0 if px0 else np.nan))
        Xl.append(im_s); Ml.append(mk_s)
    except Exception as e:
        excluded.append((os.path.relpath(p, ROOT), f"{type(e).__name__}: {e}"))

META = pd.DataFrame(rows)
X = np.stack(Xl); M = np.stack(Ml); del Xl, Ml
Y = (META["label"].values - 1).astype(int)            # class index = label - 1  (meningioma, glioma, pituitary)
PID = META["pid"].values.astype(str)
HAS = META["mask_px_orig"].values > 0                  # sample has a valid (non-empty) annotated mask
FILES = META["file"].values

# ---- verification report ----
print(f"\nsample files: {len(MAT_FILES)} | loaded: {len(META)} | excluded: {len(excluded)}")
for f, why in excluded[:20]: print("  EXCLUDED", f, "->", why)
print("MAT format used:", dict(Counter(META["fmt"])))
print("original image shapes:", dict(Counter(zip(META["H"], META["W"]))), "| image dtypes:", dict(Counter(META["img_dtype"])))
print(f"resized: X{X.shape} M{M.shape} Y{Y.shape} | mask values: {np.unique(M).tolist()} | non-binary raw masks: {int(META['mask_nonbinary'].sum())}")
assert X.shape == M.shape == (len(META), SIZE, SIZE) and set(np.unique(M).tolist()) <= {0, 1}
assert (META["H"] == META["W"]).all(), "non-square image found -> resizing to a square would distort it"
# image/mask come from the SAME .mat struct, so pairing errors are impossible by construction; shapes were asserted equal per file.
print(f"empty masks in source: {int((~HAS).sum())} | tumours that vanish after resize (mask empty at {SIZE}px): {int((M.sum((1,2)) == 0).sum())} "
      f"| area ratio resized/orig min {np.nanmin(META['area_ratio']):.3f} max {np.nanmax(META['area_ratio']):.3f}")

cnt = np.bincount(Y, minlength=NC); n_pat = len(np.unique(PID))
pat_lab = META.groupby("pid")["label"].nunique()
print("class counts (images):", dict(zip(CLASSES, cnt.tolist())), "| expected:", CFG["expected"]["per_class"])
print("unique patients:", n_pat, "| expected:", CFG["expected"]["patients"],
      "| patients with >1 distinct label:", int((pat_lab > 1).sum()))
ppc = {CLASSES[c]: len(np.unique(PID[Y == c])) for c in range(NC)}
print("patients per class:", ppc, "| expected (literature):", CFG["expected"]["patients_per_class"])
ok = (len(META) == CFG["expected"]["n"] and n_pat == CFG["expected"]["patients"]
      and dict(zip(CLASSES, cnt.tolist())) == CFG["expected"]["per_class"])
print("ALL 3,064 samples / 233 patients / per-class counts accounted for:", ok)
if not ok and STRICT_COUNTS and not SMOKE:
    raise AssertionError("Loaded data does not match the published counts - inspect the exclusion list above "
                         "(set STRICT_COUNTS=False to continue anyway).")
if ppc != CFG["expected"]["patients_per_class"]:
    print("NOTE: patients-per-class differs from the literature values; the split below uses the data as loaded.")

if SMOKE:                                              # tiny patient-grouped subset: <=10 patients/class, <=4 images/patient
    rng = np.random.RandomState(HP["seed"]); keep = []
    for c in range(NC):
        for p in rng.choice(np.unique(PID[Y == c]), size=min(10, len(np.unique(PID[Y == c]))), replace=False):
            keep += np.where(PID == p)[0][:4].tolist()
    keep = np.array(sorted(keep))
    X, M, Y, HAS, PID, FILES = X[keep], M[keep], Y[keep], HAS[keep], PID[keep], FILES[keep]
    META = META.iloc[keep].reset_index(drop=True)
    print("SMOKE subset:", len(Y), "images,", len(np.unique(PID)), "patients")
print("HAS mask supervision:", int(HAS.sum()), "/", len(HAS))

# ---- visual image/mask alignment check ----
sel = np.random.RandomState(0).choice(len(Y), size=min(6, len(Y)), replace=False)
fig, ax = plt.subplots(2, len(sel), figsize=(2.6 * len(sel), 5.4), squeeze=False)
for j, i in enumerate(sel):
    ax[0, j].imshow(X[i], cmap="gray"); ax[0, j].set_title(f"{CLASSES[Y[i]]}\nPID {PID[i]}", fontsize=8)
    ax[1, j].imshow(X[i], cmap="gray")
    if M[i].any(): ax[1, j].contour(M[i], levels=[0.5], colors="r", linewidths=1)
    for a in ax[:, j]: a.axis("off")
plt.tight_layout(); plt.savefig(f"{RES}/figures/data_alignment_check.png", dpi=120); plt.show()

In [ ]:
# ---------------- augmentation / Dataset / DataLoader (augmentation + normalisation unchanged; everything runs at SIZE=512) ----------------
# X, M, Y, HAS are already in RAM at SIZE x SIZE (cell above). Normalisation is unchanged: (x/255 - 0.449) / 0.226.
AUG = A.Compose([A.HorizontalFlip(p=0.5),
                 A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, p=0.5),
                 A.RandomBrightnessContrast(0.15, 0.15, p=0.5)])   # applied jointly to image + mask (albumentations warps masks with nearest-neighbour; masks stay {0,1})

class DS(Dataset):
    def __init__(s, idx, train): s.idx, s.train = idx, train
    def __len__(s): return len(s.idx)
    def __getitem__(s, i):
        j = s.idx[i]; x, m = X[j], M[j]
        if s.train:
            r = AUG(image=x, mask=m); x, m = r["image"], r["mask"]
        x = torch.from_numpy(np.ascontiguousarray(x)).float().div(255).sub(0.449).div(0.226).unsqueeze(0)
        m = torch.from_numpy(np.ascontiguousarray(m)).float().unsqueeze(0)
        return x, m, int(Y[j]), bool(HAS[j])

def loader(idx, train, bs=None):
    return DataLoader(DS(idx, train), batch_size=bs or HP["batch"], shuffle=train, drop_last=train,
                      num_workers=HP["workers"], pin_memory=True,
                      persistent_workers=bool(train) and HP["workers"] > 0)      # no persistent workers for the per-epoch validation loaders (saves RAM at 512 px)

# quick check: one augmented sample keeps a binary mask and the right shapes
_x, _m, _y, _h = DS(np.arange(len(Y)), True)[0]
assert _x.shape == (1, SIZE, SIZE) and _m.shape == (1, SIZE, SIZE) and set(torch.unique(_m).tolist()) <= {0.0, 1.0}   # SIZE = 512
print("dataset ok | sample:", tuple(_x.shape), tuple(_m.shape), "label", _y)

In [ ]:
# ---------------- PATIENT-LEVEL splitting: hold-out test + 5-fold stratified GROUP CV ----------------
# Groups = PID. Only StratifiedGroupKFold is used (no StratifiedShuffleSplit / StratifiedKFold). Patient separation has priority
# over class balance. To get the most balanced partition we screen SPLIT_TRIES seeded StratifiedGroupKFold candidates using ONLY
# labels / PIDs / sizes (never images or model performance) and keep the best one. The chosen seeds are saved for reproducibility.
all_idx = np.arange(len(Y))
pid_u = np.unique(PID)
pid_label = {p: int(np.bincount(Y[PID == p], minlength=NC).argmax()) for p in pid_u}     # majority label per patient
multi = [p for p in pid_u if len(np.unique(Y[PID == p])) > 1]
print(f"patients: {len(pid_u)} | patients with more than one label: {len(multi)} {multi[:5]}")
Y_STRAT = np.array([pid_label[p] for p in PID])          # patient-level label broadcast to images (== Y if every patient has 1 label)
print("Y_STRAT == Y for all images:", bool((Y_STRAT == Y).all()))

def best_group_folds(y, groups, n_splits, seed, tries, first_only=False):
    g = np.bincount(y, minlength=NC) / len(y); best = None
    for t in range(tries):
        rs = seed + t
        try:
            folds = [te for _, te in StratifiedGroupKFold(n_splits, shuffle=True, random_state=rs)
                     .split(np.zeros(len(y)), y, groups)]
        except ValueError:
            continue
        use = folds[:1] if first_only else folds
        sc = float(np.mean([np.abs(np.bincount(y[f], minlength=NC) / max(len(f), 1) - g).sum()
                            + 2 * abs(len(f) / len(y) - 1 / n_splits) for f in use]))
        if best is None or sc < best[0]: best = (sc, rs, folds)
    if best is None: raise RuntimeError("could not build a group split (too few patients?)")
    return best[2], best[1], best[0]

# Stage 1: hold-out test patients (~test_frac of patients)
HO_SPLITS = int(round(1 / HP["test_frac"]))
ho_folds, ho_seed, ho_score = best_group_folds(Y_STRAT, PID, HO_SPLITS, HP["seed"], SPLIT_TRIES, first_only=True)
te_idx = np.sort(ho_folds[0]); tv_idx = np.setdiff1d(all_idx, te_idx)

# Stage 2: K-fold stratified group CV on the train/val patients
cv_rel, cv_seed, cv_score = best_group_folds(Y_STRAT[tv_idx], PID[tv_idx], HP["n_folds"], HP["seed"], SPLIT_TRIES)
FOLDS = []
for va_rel in cv_rel:
    va = tv_idx[np.sort(va_rel)]; tr = np.setdiff1d(tv_idx, va); FOLDS.append((tr, va))
print(f"hold-out: StratifiedGroupKFold(n_splits={HO_SPLITS}) random_state={ho_seed} balance-score={ho_score:.4f}")
print(f"CV      : StratifiedGroupKFold(n_splits={HP['n_folds']}) random_state={cv_seed} balance-score={cv_score:.4f}")

# ---------------- split lock: the baseline (256 px) patient assignment must not change with the resolution ----------------
# The algorithm above uses only labels/PIDs/seeds, so it is resolution-independent. If the baseline run's split_info.json is available
# (attach the baseline output as a Kaggle dataset, or set CFG["reference_split_json"]) it is used AS the split and compared with the regenerated one.
def find_reference_split():
    if CFG.get("reference_split_json"): return CFG["reference_split_json"] if os.path.isfile(CFG["reference_split_json"]) else None
    for root in ("/kaggle/input", "/kaggle/working", "."):
        hits = sorted(glob.glob(f"{root}/**/Figshare_Experiment_Results/split_info.json", recursive=True))
        if hits: return hits[0]
    return None
SPLIT_SOURCE = "regenerated deterministically (same algorithm, seeds and sample order as the baseline notebook); baseline split_info.json not found"
ref_path = None if SMOKE else find_reference_split()
if ref_path:
    ref = json.load(open(ref_path)); ref_te = set(ref["holdout"]["test_pids"]); ref_val = [set(f["val_pids"]) for f in ref["cv"]["folds"]]
    ref_all = ref_te.union(*ref_val)
    if ref_all == set(pid_u) and not (ref_te & set().union(*ref_val)) and len(ref_val) == HP["n_folds"]:
        te_ref = np.sort(np.where(np.isin(PID, list(ref_te)))[0]); tv_ref = np.setdiff1d(all_idx, te_ref); folds_ref = []
        for pv in ref_val:
            va = np.sort(np.where(np.isin(PID, list(pv)))[0]); folds_ref.append((np.setdiff1d(tv_ref, va), va))
        same = np.array_equal(te_ref, te_idx) and all(np.array_equal(a[1], b[1]) for a, b in zip(folds_ref, FOLDS))
        print(f"baseline split found: {ref_path} | regenerated split identical to it: {same}")
        te_idx, tv_idx, FOLDS = te_ref, tv_ref, folds_ref
        ho_seed, cv_seed = ref["holdout"]["selected_random_state"], ref["cv"]["selected_random_state"]
        ho_score, cv_score = ref["holdout"]["balance_score"], ref["cv"]["balance_score"]
        SPLIT_SOURCE = f"loaded from the baseline run: {ref_path} (regenerated split identical: {same})"
    else:
        print(f"WARNING: {ref_path} does not match this dataset's patients/folds -> ignored (regenerated split is used)")
if not SMOKE:
    got = dict(n_images=int(len(te_idx)), n_patients=len(set(PID[te_idx])))
    if got != CFG["expected_test"]:
        msg = f"hold-out test set is {got}, but the baseline test set is {CFG['expected_test']} -> this is NOT the baseline split"
        if STRICT_SPLIT: raise AssertionError(msg + " (attach the baseline split_info.json or set STRICT_SPLIT=False)")
        print("WARNING:", msg)
print("SPLIT SOURCE:", SPLIT_SOURCE)

# ---------------- explicit leakage assertions ----------------
ps = lambda idx: set(PID[idx])
P_te, P_tv = ps(te_idx), ps(tv_idx)
assert len(P_tv & P_te) == 0, "LEAK: a patient is in both train/val and test"
assert len(set(tv_idx) & set(te_idx)) == 0 and len(tv_idx) + len(te_idx) == len(Y)
assert P_tv | P_te == set(pid_u)
val_sets = []
for k, (tr, va) in enumerate(FOLDS):
    Ptr, Pva = ps(tr), ps(va)
    assert len(Ptr & Pva) == 0, f"LEAK fold {k}: train PID intersect validation PID != 0"
    assert len(Ptr & P_te) == 0, f"LEAK fold {k}: train PID intersect test PID != 0"
    assert len(Pva & P_te) == 0, f"LEAK fold {k}: validation PID intersect test PID != 0"
    assert len(set(tr) & set(va)) == 0 and len(tr) + len(va) == len(tv_idx)
    assert Ptr | Pva == P_tv
    val_sets.append(Pva)
for i in range(len(val_sets)):
    for j in range(i + 1, len(val_sets)):
        assert len(val_sets[i] & val_sets[j]) == 0, f"LEAK: patient in validation of folds {i} and {j}"
assert set().union(*val_sets) == P_tv and sum(len(v) for v in val_sets) == len(P_tv)
print("PATIENT-LEVEL LEAKAGE CHECKS PASSED: train∩val = train∩test = val∩test = 0 patients; every train/val patient is in exactly one validation fold.")

# ---------------- report patients / images / class distribution of every split ----------------
def split_stats(name, idx):
    pids = ps(idx); ci = np.bincount(Y[idx], minlength=NC); cp = np.bincount([pid_label[p] for p in pids], minlength=NC)
    d = dict(split=name, n_patients=len(pids), n_images=len(idx))
    d.update({f"img_{c}": int(n) for c, n in zip(CLASSES, ci)}); d.update({f"pat_{c}": int(n) for c, n in zip(CLASSES, cp)})
    return d
srows = [split_stats("ALL", all_idx), split_stats("train+val (hold-in)", tv_idx), split_stats("TEST (hold-out)", te_idx)]
for k, (tr, va) in enumerate(FOLDS):
    srows += [split_stats(f"fold{k} train", tr), split_stats(f"fold{k} val", va)]
SPLIT_SUMMARY = pd.DataFrame(srows)
print(); print(SPLIT_SUMMARY.to_string(index=False))
for r in srows:
    miss = [c for c in CLASSES if r[f"img_{c}"] == 0]
    if miss: print(f"WARNING: split '{r['split']}' has no images of class(es) {miss} (AUC for them will be NaN)")

# ---------------- save split information (seed + patient IDs per split) ----------------
fold_of_img = np.full(len(Y), -1)
for k, (tr, va) in enumerate(FOLDS): fold_of_img[va] = k
part = np.where(np.isin(all_idx, te_idx), "test", "trainval")
pd.DataFrame(dict(sample_index=all_idx, file=FILES, pid=PID, label=Y + 1, cls=[CLASSES[c] for c in Y],
                  partition=part, cv_val_fold=fold_of_img)).to_csv(f"{RES}/image_split_assignment.csv", index=False)
pd.DataFrame([dict(pid=p, cls=CLASSES[pid_label[p]], n_images=int((PID == p).sum()),
                   partition="test" if p in P_te else "trainval",
                   cv_val_fold=int(fold_of_img[np.where(PID == p)[0][0]])) for p in pid_u]
             ).sort_values(["partition", "cv_val_fold", "pid"]).to_csv(f"{RES}/patient_split_assignment.csv", index=False)
SPLIT_SUMMARY.to_csv(f"{RES}/split_summary.csv", index=False)
json.dump(dict(random_seed=HP["seed"], dataset=DATASET, split_source=SPLIT_SOURCE, resolution=SIZE, n_images=int(len(Y)), n_patients=int(len(pid_u)),
               holdout=dict(n_splits=HO_SPLITS, selected_random_state=int(ho_seed), balance_score=ho_score,
                            n_candidates_screened=SPLIT_TRIES, test_pids=sorted(P_te)),
               cv=dict(n_folds=HP["n_folds"], selected_random_state=int(cv_seed), balance_score=cv_score,
                       folds=[dict(fold=k, train_pids=sorted(ps(tr)), val_pids=sorted(ps(va))) for k, (tr, va) in enumerate(FOLDS)]),
               note="reproduce with StratifiedGroupKFold(n_splits, shuffle=True, random_state=selected_random_state) on "
                    "(labels broadcast per patient, groups=PID); see image_split_assignment.csv for the exact per-image assignment"),
          open(f"{RES}/split_info.json", "w"), indent=1)
print("saved split files to", RES)

In [ ]:
# ---------------- FinalNet-ASPP-FPN-512 ----------------
# Flow: MRI -> EfficientNet-B0 encoder (strides 2,4,8,16,32) -> ASPP on the stride-32 bottleneck (320 ch, residual)
#       -> FPN top-down fusion over ALL five encoder stages (seeded by the ASPP output)
#       -> fused skips (original skip + 1x1-adapted FPN level) + ASPP bottleneck -> UNCHANGED smp U-Net decoder (scSE)
#       -> main seg head + 3 aux heads (decoder blocks 1,2,3) ; classifier on (ASPP-bottleneck, FPN-fused stride-16 skip) with the soft mask.
ARCH = dict(name="FinalNet-ASPP-FPN-512", encoder="tu-efficientnet_b0", decoder_channels=(256, 128, 64, 32, 16),
            aspp_branch_ch=128, aspp_rates=(2, 4, 6), aspp_dropout=0.1, aspp_residual=True,
            fpn_ch=64, fpn_adapter_zero_init=True)

class ShapeError(RuntimeError):
    """Raised with an explicit description when feature-map shapes do not line up."""

def cbr(cin, cout, k=1, d=1):
    """Conv -> BatchNorm -> ReLU ('same' padding, optional dilation)."""
    return nn.Sequential(nn.Conv2d(cin, cout, k, padding=d * (k // 2), dilation=d, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class ASPP(nn.Module):
    """Atrous Spatial Pyramid Pooling for the stride-32 bottleneck (16x16 map at 512 px input).
    Branches: 1x1 | 3x3 dilated (rates) | global average pooling. Concatenate -> 1x1 projection to `cout` -> BN -> ReLU -> Dropout2d.
    cout == bottleneck channels (320), so the decoder and the classification head keep their input widths.
    Rates (2,4,6): on a 16x16 map a 3x3 kernel with rate r reaches r pixels from the centre; r=6 spans ~75 % of the map, r>8 would only read
    zero padding. The global branch has no BatchNorm (a 1x1 map would break BN statistics with tiny micro-batches)."""
    def __init__(self, cin, cout, branch_ch=128, rates=(2, 4, 6), drop=0.1, residual=True):
        super().__init__()
        self.rates, self.residual = tuple(rates), bool(residual) and cin == cout
        self.b0 = cbr(cin, branch_ch, 1)
        self.bd = nn.ModuleList([cbr(cin, branch_ch, 3, d=r) for r in self.rates])
        self.gp = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(cin, branch_ch, 1), nn.ReLU(inplace=True))
        self.proj = nn.Sequential(nn.Conv2d(branch_ch * (2 + len(self.rates)), cout, 1, bias=False), nn.BatchNorm2d(cout),
                                  nn.ReLU(inplace=True), nn.Dropout2d(drop))
    def forward(self, x):
        h, w = x.shape[-2:]
        if max(self.rates) > min(h, w) // 2:
            raise ShapeError(f"ASPP dilation rates {self.rates} are too large for the {h}x{w} bottleneck (max useful rate {min(h, w) // 2})")
        y = [self.b0(x)] + [b(x) for b in self.bd] + [self.gp(x).expand(-1, -1, h, w)]
        y = self.proj(torch.cat(y, 1))
        return x + y if self.residual else y

class FPNFusion(nn.Module):
    """FPN-style top-down fusion over the encoder stages (strides 2,4,8,16,32; channels `enc_ch`, shallow -> deep).
    M_deep = lateral1x1(deepest);  M_i = lateral1x1(stage_i) + upsample(M_{i+1})   (nearest, to the exact size of stage_i)
    P_i = smooth3x3(M_i)  (conv-BN-ReLU, `ch` channels).
    To plug into the UNCHANGED decoder, each P_i is mapped back to the width of its stage by a 1x1 conv + BN (zero-initialised gamma ->
    the network starts as 'baseline + ASPP' and learns how much multi-scale context to inject) and ADDED to the original stage, so the
    high-resolution encoder features are kept intact. The deepest entry (the ASPP output) is returned unchanged."""
    def __init__(self, enc_ch, ch=64, zero_init=True):
        super().__init__()
        self.enc_ch = tuple(enc_ch)
        self.lat = nn.ModuleList([nn.Conv2d(c, ch, 1) for c in self.enc_ch])
        self.smooth = nn.ModuleList([cbr(ch, ch, 3) for _ in self.enc_ch])
        self.adapt = nn.ModuleList([nn.Sequential(nn.Conv2d(ch, c, 1, bias=False), nn.BatchNorm2d(c)) for c in self.enc_ch[:-1]])
        if zero_init:
            for a in self.adapt: nn.init.zeros_(a[1].weight)
    def forward(self, stages):
        n = len(self.lat)
        if len(stages) != n: raise ShapeError(f"FPN expects {n} encoder stages, got {len(stages)}")
        for i, (s, c) in enumerate(zip(stages, self.enc_ch)):
            if s.shape[1] != c: raise ShapeError(f"FPN stage {i}: expected {c} channels, got {s.shape[1]} (shape {tuple(s.shape)})")
            if i + 1 < n and (s.shape[-2] != 2 * stages[i + 1].shape[-2] or s.shape[-1] != 2 * stages[i + 1].shape[-1]):
                raise ShapeError(f"FPN stage {i} {tuple(s.shape)} is not exactly 2x the size of stage {i + 1} {tuple(stages[i + 1].shape)}")
        m = self.lat[-1](stages[-1]); ps = [None] * n; ps[-1] = self.smooth[-1](m)
        for i in range(n - 2, -1, -1):
            m = self.lat[i](stages[i]) + F.interpolate(m, size=stages[i].shape[-2:], mode="nearest")
            ps[i] = self.smooth[i](m)
        return [s + self.adapt[i](ps[i]) for i, s in enumerate(stages[:-1])] + [stages[-1]]

def build_smp_unet(enc, pretrained, dch):
    """Same smp.Unet as the baseline. With pretrained=True the call is identical to the baseline (ImageNet weights, from the HF/timm cache or
    the network). If that fails (Internet OFF and nothing cached) a local weight file is searched; if none exists we RAISE - the encoder is
    never silently left randomly initialised."""
    global PRETRAINED_SOURCE
    kw = dict(in_channels=3, classes=1, decoder_channels=dch, decoder_attention_type="scse")
    if not pretrained:
        PRETRAINED_SOURCE = "none (pretrained=False, weights come from a checkpoint)"; return smp.Unet(enc, encoder_weights=None, **kw)
    try:
        base = smp.Unet(enc, encoder_weights=True, **kw); PRETRAINED_SOURCE = "smp/timm pretrained (cache or network)"; return base
    except Exception as e:
        err = f"{type(e).__name__}: {str(e)[:200]}"
    cands = [p for ext in ("pth", "pt", "bin", "safetensors") for p in glob.glob(f"/kaggle/input/**/*efficientnet_b0*.{ext}", recursive=True)]
    cands += [p for ext in ("pth", "pt", "bin", "safetensors") for p in glob.glob(f"/kaggle/input/**/*efficientnet*b0*/*.{ext}", recursive=True)]
    if not cands:
        raise RuntimeError(f"Pretrained EfficientNet-B0 weights could not be loaded ({err}).\n"
                           "Offline fix: with Internet ON once, run `timm.create_model('efficientnet_b0', pretrained=True)` and save "
                           "`torch.save(m.state_dict(), 'efficientnet_b0.pth')` (or the HF cache folder) into a Kaggle Dataset, attach it, and re-run. "
                           "The encoder is NOT replaced by a randomly initialised one.")
    path = sorted(set(cands))[0]
    if path.endswith(".safetensors"):
        from safetensors.torch import load_file; sd = load_file(path)
    else:
        sd = torch.load(path, map_location="cpu"); sd = sd.get("state_dict", sd) if isinstance(sd, dict) else sd.state_dict()
    base = smp.Unet(enc, encoder_weights=None, **kw)
    res = base.encoder.model.load_state_dict(sd, strict=False)
    bad = [k for k in res.missing_keys if k.startswith(("conv_stem", "bn1", "blocks"))]
    if bad: raise RuntimeError(f"local weight file {path} does not match the EfficientNet-B0 encoder (missing e.g. {bad[:3]})")
    print(f"loaded local pretrained encoder weights: {path} | missing {len(res.missing_keys)} (classifier/head only) | unexpected {len(res.unexpected_keys)}")
    PRETRAINED_SOURCE = f"local file {path}"; return base
PRETRAINED_SOURCE = "unknown"

class FinalNet(nn.Module):
    def __init__(self, num_classes, pretrained=True, enc="tu-efficientnet_b0", arch=None):
        super().__init__()
        a = dict(ARCH); a.update(arch or {}); self.arch = a
        dch = a["decoder_channels"]
        base = build_smp_unet(enc, pretrained, dch)
        self.encoder, self.decoder, self.seg_head = base.encoder, base.decoder, base.segmentation_head
        self.aux_ids = (1, 2, 3)                                   # 3 auxiliary 1x1 heads
        self.aux_heads = nn.ModuleList([nn.Conv2d(dch[i], 1, 1) for i in self.aux_ids])
        self._f = {}
        for i in self.aux_ids:
            self.decoder.blocks[i].register_forward_hook(self._mk(i))
        ps = inspect.signature(self.decoder.forward).parameters
        self._star = any(p.kind == p.VAR_POSITIONAL for p in ps.values())   # smp-version compatibility
        ch = list(self.encoder.out_channels)                       # (3, 16, 24, 40, 112, 320) for EfficientNet-B0
        if len(ch) != 6: raise ShapeError(f"expected 6 encoder feature levels (input + 5 stages), got channels {ch}")
        self.aspp = ASPP(ch[-1], ch[-1], a["aspp_branch_ch"], a["aspp_rates"], a["aspp_dropout"], a["aspp_residual"])
        self.fpn = FPNFusion(ch[1:], a["fpn_ch"], a["fpn_adapter_zero_init"])
        c32, c16 = ch[-1], ch[-2]                                  # classifier input widths are unchanged (320*2 + 112)
        self.head = nn.Sequential(nn.Linear(c32 * 2 + c16, 256), nn.ReLU(True), nn.Dropout(0.3),
                                  nn.Linear(256, num_classes))

    def _mk(self, i):
        def hook(mod, inp, out): self._f[i] = out
        return hook

    def fuse(self, feats):
        """encoder features -> [input, FPN-fused skips (strides 2,4,8,16), ASPP bottleneck (stride 32)]."""
        if len(feats) != 6: raise ShapeError(f"encoder returned {len(feats)} feature maps, expected 6: {[tuple(f.shape) for f in feats]}")
        stages = list(feats[1:]); stages[-1] = self.aspp(stages[-1])
        return [feats[0]] + self.fpn(stages)

    def forward(self, x):
        feats = self.fuse(self.encoder(x))
        d = self.decoder(*feats) if self._star else self.decoder(feats)
        seg = self.seg_head(d)
        aux = [h(self._f[i]) for h, i in zip(self.aux_heads, self.aux_ids)]
        self._f.clear()
        m = torch.sigmoid(seg.float())                              # soft mask: no threshold, no detach
        f32, f16 = feats[-1].float(), feats[-2].float()             # ASPP bottleneck (stride 32) and FPN-fused skip (stride 16)
        def wpool(f):
            mm = F.interpolate(m, size=f.shape[-2:], mode="bilinear", align_corners=False)
            return (f * mm).sum((2, 3)) / (mm.sum((2, 3)) + 1e-6)
        z = torch.cat([f32.mean((2, 3)), wpool(f32), wpool(f16)], 1)
        return dict(seg=seg, aux=aux, cls=self.head(z))

# ---------------- losses (unchanged: Tversky + BCE, CE with label smoothing in the training cell, learned uncertainty weights) ----------------
def tversky(logit, t, alpha=0.7, eps=1.0):
    p = torch.sigmoid(logit.float())
    tp = (p * t).sum(); fn = ((1 - p) * t).sum(); fp = (p * (1 - t)).sum()
    return 1 - (tp + eps) / (tp + alpha * fn + (1 - alpha) * fp + eps)

def seg_loss(out, tgt, valid):
    if valid.sum() == 0: return None
    tot = 0; outs = [out["seg"]] + out["aux"]
    for o in outs:
        o, t = o[valid], tgt[valid]
        if o.shape[-2:] != t.shape[-2:]:
            o = F.interpolate(o.float(), size=t.shape[-2:], mode="bilinear", align_corners=False)
        tot = tot + tversky(o, t) + F.binary_cross_entropy_with_logits(o.float(), t)
    return tot / len(outs)

class Uncertainty(nn.Module):                                      # learned task weights (Kendall et al.)
    def __init__(s): super().__init__(); s.s = nn.Parameter(torch.zeros(2))
    def forward(s, ls, lc):
        t = torch.exp(-s.s[1]) * lc + s.s[1]
        if ls is not None: t = t + torch.exp(-s.s[0]) * ls + s.s[0]
        return t

class EMA:
    def __init__(s, model, decay):
        s.m = FinalNet(NC, pretrained=False).to(dev); s.m.load_state_dict(model.state_dict()); s.m.eval()
        for p in s.m.parameters(): p.requires_grad_(False)
        s.d, s.n = decay, 0
    @torch.no_grad()
    def update(s, model):
        s.n += 1; d = min(s.d, (1 + s.n) / (10 + s.n))
        for e, p in zip(s.m.state_dict().values(), model.state_dict().values()):
            if e.dtype.is_floating_point: e.mul_(d).add_(p.detach(), alpha=1 - d)
            else: e.copy_(p)

In [ ]:
import gc
# ---------------- ARCHITECTURE VALIDATION + SMOKE TEST (one forward/backward only; NO training is started here) ----------------
def n_params(m, trainable=False): return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable))
def chk(cond, msg):
    if not cond: raise ShapeError(msg)

def diagnose_shapes(model, x):
    """Print every fusion stage so a dimension mismatch can be located."""
    with torch.no_grad():
        try:
            feats = model.encoder(x); print("  encoder stages :", [tuple(f.shape) for f in feats])
            st = list(feats[1:]); st[-1] = model.aspp(st[-1]); print("  ASPP output    :", tuple(st[-1].shape))
            print("  FPN outputs    :", [tuple(f.shape) for f in model.fpn(st)])
        except Exception as e: print("  (diagnosis stopped:", type(e).__name__, str(e)[:200], ")")

def make_batch(B):
    x = torch.randn(B, 3, SIZE, SIZE, device=dev)
    t = (torch.rand(B, 1, SIZE, SIZE, device=dev) > 0.8).float(); y = torch.randint(0, NC, (B,), device=dev)
    return x, t, y

def train_step_check(model, B):
    """One forward + backward with the ACTUAL losses (seg_loss = Tversky+BCE over main+3 aux, CE with label smoothing 0.1, uncertainty weights)."""
    x, t, y = make_batch(B); unc = Uncertainty().to(dev); ce = nn.CrossEntropyLoss(label_smoothing=0.1)
    model.zero_grad(set_to_none=True); model.train()
    with torch.autocast(dev.type, enabled=dev.type == "cuda"):
        out = model(x)
    ls = seg_loss(out, t, torch.ones(B, dtype=torch.bool, device=dev)); lc = ce(out["cls"].float(), y)
    loss = unc(ls, lc); chk(torch.isfinite(loss).item(), f"training loss is not finite: {loss.item()}")
    loss.backward(); return loss.item()

def is_oom(e): return isinstance(e, torch.cuda.OutOfMemoryError) if hasattr(torch.cuda, "OutOfMemoryError") else "out of memory" in str(e)

# ---- 1-3: instantiate (pretrained encoder if obtainable -> also tests the offline path), print architecture + parameter counts ----
try:
    smoke = FinalNet(NC, pretrained=True).to(dev); PRE_SRC = PRETRAINED_SOURCE; print("pretrained encoder weights:", PRE_SRC)
except Exception as e:
    print("!" * 100, f"\nWARNING: pretrained encoder weights are NOT available here -> training cell would fail:\n{e}\n" + "!" * 100)
    smoke = FinalNet(NC, pretrained=False).to(dev); PRE_SRC = "UNAVAILABLE (offline and no local weights) - shape test only"
print(smoke); print()
parts = dict(encoder=smoke.encoder, aspp=smoke.aspp, fpn=smoke.fpn, decoder=smoke.decoder, seg_head=smoke.seg_head, aux_heads=smoke.aux_heads, cls_head=smoke.head)
PARAMS = {k: n_params(v) for k, v in parts.items()}; PARAMS["total"] = n_params(smoke); PARAMS["trainable"] = n_params(smoke, True)
assert sum(PARAMS[k] for k in parts) == PARAMS["total"]
for k, v in PARAMS.items(): print(f"  params {k:10s}: {v:>12,d} ({v / 1e6:.3f} M)")

# ---- 4-8: dummy [2,3,512,512] forward: shapes + finiteness ----
smoke.eval(); x2 = torch.randn(2, 3, SIZE, SIZE, device=dev)
try:
    with torch.no_grad(), torch.autocast(dev.type, enabled=dev.type == "cuda"):
        o = smoke(x2)
except (RuntimeError, ValueError) as e:
    print("forward failed - stage shapes:"); diagnose_shapes(smoke, x2)
    raise ShapeError(f"forward pass failed for input {tuple(x2.shape)}: {e}") from e
aux_shapes = [tuple(a.shape) for a in o["aux"]]
print("main seg:", tuple(o["seg"].shape), "| aux:", aux_shapes, "| cls:", tuple(o["cls"].shape))
chk(tuple(o["seg"].shape) == (2, 1, SIZE, SIZE), f"main segmentation output must be (2,1,{SIZE},{SIZE}), got {tuple(o['seg'].shape)}")
chk(len(o["aux"]) == 3, f"expected 3 auxiliary heads, got {len(o['aux'])}")
hs = [s[-1] for s in aux_shapes]
chk(all(s[0] == 2 and s[1] == 1 and s[2] == s[3] for s in aux_shapes) and all(SIZE % h == 0 and h < SIZE for h in hs) and hs == sorted(hs),
    f"auxiliary outputs must be square, increasing and divide {SIZE}: {aux_shapes}")
print(f"  auxiliary heads at strides {[SIZE // h for h in hs]} (seg_loss bilinearly resizes their logits to {SIZE}x{SIZE}; masks are never resized)")
chk(tuple(o["cls"].shape) == (2, NC) and NC == 3, f"classification output must be (2,3), got {tuple(o['cls'].shape)}")
for name, t in [("seg", o["seg"]), ("cls", o["cls"])] + [(f"aux{i}", a) for i, a in enumerate(o["aux"])]:
    chk(bool(torch.isfinite(t.float()).all()), f"non-finite values in output '{name}'")
btl = SIZE // 32; chk(max(ARCH["aspp_rates"]) <= btl // 2, f"ASPP rates {ARCH['aspp_rates']} too large for the {btl}x{btl} bottleneck")
print(f"  bottleneck {btl}x{btl}; ASPP rates {ARCH['aspp_rates']} (+1x1, +global pooling); FPN width {ARCH['fpn_ch']}")

# ---- 9-10: one forward+backward with the actual losses on [2,3,512,512]; peak CUDA memory ----
if dev.type == "cuda": torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
loss_val = train_step_check(smoke, 2)
bad = [n for n, p in smoke.named_parameters() if p.grad is not None and not bool(torch.isfinite(p.grad).all())]
chk(not bad, f"non-finite gradients in: {bad[:5]}")
g = lambda mod: float(sum(p.grad.float().norm() ** 2 for p in mod.parameters() if p.grad is not None) ** 0.5)
chk(g(smoke.aspp) > 0, "ASPP received no gradient (not connected to the loss)")
chk(all(float(a[1].weight.grad.abs().sum()) > 0 for a in smoke.fpn.adapt), "an FPN adapter received no gradient (FPN not connected to the decoder)")
print(f"one train step OK | loss {loss_val:.4f} | grad-norm ASPP {g(smoke.aspp):.3e}, FPN {g(smoke.fpn):.3e} "
      f"(FPN lateral/smoothing grads are exactly 0 at init because the adapters are zero-initialised; they become non-zero after the first optimiser step)")
PEAK = {}
if dev.type == "cuda": PEAK["batch2_gib"] = torch.cuda.max_memory_allocated() / 2**30; print(f"peak CUDA memory, one train step, batch 2 @ {SIZE}x{SIZE}: {PEAK['batch2_gib']:.2f} GiB")
else: print("CUDA not available -> peak memory not measured (shape / loss / gradient checks ran on CPU)")

# ---- memory probe: largest micro-batch (divisor of the effective batch) that fits; effective batch stays HP['batch'] ----
if dev.type == "cuda":
    tot = torch.cuda.get_device_properties(0).total_memory; cands = [b for b in range(HP["batch"], 1, -1) if HP["batch"] % b == 0]
    probe = {}
    for B in (cands if AUTO_MICRO_BATCH else [HP["micro_batch"]]):
        del smoke; smoke = FinalNet(NC, pretrained=False).to(dev); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        try:
            train_step_check(smoke, B); probe[B] = torch.cuda.max_memory_allocated() / 2**30
            print(f"  micro-batch {B:2d}: peak {probe[B]:.2f} GiB of {tot / 2**30:.1f} GiB")
            if AUTO_MICRO_BATCH and probe[B] <= 0.80 * tot / 2**30: HP["micro_batch"] = B; break
        except Exception as e:
            if not is_oom(e): raise
            print(f"  micro-batch {B:2d}: CUDA out of memory"); probe[B] = float("inf")
        finally:
            gc.collect(); torch.cuda.empty_cache()
    chk(any(np.isfinite(v) for v in probe.values()), "no micro-batch >= 2 fits on this GPU at 512x512; reduce ARCH['fpn_ch'] / enable checkpointing")
    if not AUTO_MICRO_BATCH: chk(np.isfinite(probe[HP["micro_batch"]]), f"configured micro_batch {HP['micro_batch']} does not fit; set AUTO_MICRO_BATCH=True")
    PEAK["probe_gib"] = {k: (v if np.isfinite(v) else None) for k, v in probe.items()}
    check_batch_cfg()
print(f"==> TRAINING CONFIG: effective batch {HP['batch']} = micro-batch {HP['micro_batch']} x gradient-accumulation {HP['accum']} "
      f"(baseline: batch 16, no accumulation). Optimiser steps, LR schedule length, EMA updates/step and clipping are unchanged.")

# ---- save architecture summary + model configuration ----
with open(f"{RES}/architecture_summary.txt", "w") as f:
    f.write(f"{ARCH['name']}\ninput [B,3,{SIZE},{SIZE}] -> main seg [B,1,{SIZE},{SIZE}], aux {aux_shapes}, cls [B,{NC}]\n\n")
    f.write("\n".join(f"params {k}: {v:,d}" for k, v in PARAMS.items()) + "\n\n" + str(smoke) + "\n")
json.dump(jsonable(dict(arch=ARCH, size=SIZE, encoder_out_channels=list(smoke.encoder.out_channels), bottleneck=f"{btl}x{btl}",
                        output_shapes=dict(seg=[2, 1, SIZE, SIZE], aux=aux_shapes, cls=[2, NC]), params=PARAMS,
                        pretrained_source=PRE_SRC, effective_batch=HP["batch"], micro_batch=HP["micro_batch"], accumulation=HP["accum"],
                        peak_cuda_gib=PEAK)), open(f"{RES}/model_config.json", "w"), indent=2)
print("saved architecture_summary.txt and model_config.json to", RES)
del smoke, x2, o
if dev.type == "cuda": torch.cuda.empty_cache()
if SMOKE_TEST_ONLY: raise KeyboardInterrupt("SMOKE_TEST_ONLY=True: architecture validation finished - no training was started. Set it to False and run the remaining cells.")

In [ ]:
# ---------------- metrics (numpy): validation metrics used in training + full Table A / Table B ----------------
SEG_THR = 0.5                                   # ONE threshold everywhere (validation, test, ensemble), applied after probability averaging
HD_PENALTY = float(np.hypot(SIZE, SIZE))        # image diagonal in pixels @ 512x512 (=724.1; was 361.7 at 256) - used only for the *penalised* HD95 variant; definition unchanged

def cls_metrics(y, prob):                        # unchanged: used for validation / early stopping
    pred = prob.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    return dict(acc=accuracy_score(y, pred), prec=p, rec=r, f1=f)

def seg_stats(prob, gt, thr=SEG_THR):
    """Per-image binary segmentation statistics. prob (N,H,W) probabilities, gt (N,H,W) {0,1}.
    Conventions for degenerate cases (nothing is dropped):
      both pred and GT empty -> Dice = IoU = precision = recall = 1 (correct 'nothing');
      GT non-empty, pred empty (failed detection) -> Dice = IoU = recall = 0, precision = 0;
      GT empty, pred non-empty (false alarm)      -> Dice = IoU = precision = 0, recall = 1 (vacuous);
      specificity is 1 when there are no negative pixels."""
    pred = prob > thr; g = gt > 0.5
    tp = (pred & g).sum((1, 2)).astype(np.float64); fp = (pred & ~g).sum((1, 2)).astype(np.float64)
    fn = (~pred & g).sum((1, 2)).astype(np.float64); tn = (~pred & ~g).sum((1, 2)).astype(np.float64)
    sp, sg = pred.sum((1, 2)).astype(np.float64), g.sum((1, 2)).astype(np.float64)
    both_empty = (sp == 0) & (sg == 0)
    return dict(
        dice=np.where(both_empty, 1.0, 2 * tp / np.maximum(sp + sg, 1)),
        iou=np.where(both_empty, 1.0, tp / np.maximum(sp + sg - tp, 1)),
        precision=np.where(sp > 0, tp / np.maximum(sp, 1), np.where(sg == 0, 1.0, 0.0)),
        recall=np.where(sg > 0, tp / np.maximum(sg, 1), 1.0),
        specificity=np.where((tn + fp) > 0, tn / np.maximum(tn + fp, 1), 1.0),
        tp=tp, fp=fp, fn=fn, tn=tn, pred_px=sp, gt_px=sg)

def dice_imgs(mprob, gt, has=None):
    """Per-image Dice (macro over images) used for validation. Includes every image that has mask supervision
    (`has`); images with an empty GT are NOT dropped (empty/empty -> 1, false alarm -> 0)."""
    d = seg_stats(mprob, gt)["dice"]
    return d if has is None else d[np.asarray(has, bool)]

_CROSS = ndi.generate_binary_structure(2, 1)
def _surface(m): return m & ~ndi.binary_erosion(m, structure=_CROSS, border_value=0)

def hd95_single(pred, gt):
    """95th-percentile Hausdorff distance in PIXELS (MedPy-style: max of the two directed 95th percentiles of
    surface-to-surface distances). Both empty -> 0. Exactly one empty -> NaN (undefined; counted & reported)."""
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() and not gt.any(): return 0.0
    if not pred.any() or not gt.any():  return float("nan")
    sp, sg = _surface(pred), _surface(gt)
    d_to_gt = ndi.distance_transform_edt(~sg)[sp]; d_to_pred = ndi.distance_transform_edt(~sp)[sg]
    return float(max(np.percentile(d_to_gt, 95), np.percentile(d_to_pred, 95)))

def full_cls_report(y, P, classes=CLASSES):
    """Table A. Macro precision/recall/F1 over ALL classes (zero_division=0); one-vs-rest AUC per class
    (NaN if a class is absent or is the only class in the split) and macro-AUC = mean over the classes where it is defined."""
    y = np.asarray(y); K = len(classes); pred = P.argmax(1); labs = list(range(K))
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=labs, average="macro", zero_division=0)
    pc, rc, fc, sc = precision_recall_fscore_support(y, pred, labels=labs, average=None, zero_division=0)
    aucs = []
    for c in labs:
        yb = (y == c)
        aucs.append(float(roc_auc_score(yb, P[:, c])) if 0 < yb.sum() < len(yb) else float("nan"))
    valid = [a for a in aucs if not np.isnan(a)]
    macro_auc = float(np.mean(valid)) if valid else float("nan")
    if len(valid) == K and K > 2:                # cross-check against scikit-learn's multiclass OvR macro AUC
        Pn = P / P.sum(1, keepdims=True)
        assert abs(macro_auc - roc_auc_score(y, Pn, multi_class="ovr", average="macro", labels=labs)) < 1e-6
    cm = confusion_matrix(y, pred, labels=labs)
    overall = dict(n=int(len(y)), accuracy=float(accuracy_score(y, pred)), macro_precision=float(p), macro_recall=float(r),
                   macro_f1=float(f), macro_auc_ovr=macro_auc, n_classes_with_auc=len(valid))
    per_class = pd.DataFrame(dict(cls=classes, precision=pc, recall=rc, f1=fc, auc_ovr=aucs, support=sc.astype(int)))
    return overall, per_class, cm

def full_seg_report(prob, gt, y, classes=CLASSES, thr=SEG_THR, show_progress=True):
    """Table B + per-class Dice on ALL images (nothing excluded). Dice/IoU/precision/recall/specificity are reported both as
    the mean of per-image values (macro over images) and globally pooled over all pixels. HD95 is per image, in pixels."""
    s = seg_stats(prob, gt, thr); N = len(y); pred = prob > thr; g = gt > 0.5
    it = tqdm(range(N), desc="HD95", leave=False) if show_progress else range(N)
    hd = np.array([hd95_single(pred[i], g[i]) for i in it])
    hd_def = hd[~np.isnan(hd)]
    pen = np.where(np.isnan(hd), HD_PENALTY, hd)
    def pooled(sel):
        TP, FP, FN, TN = (s[k][sel].sum() for k in ("tp", "fp", "fn", "tn"))
        dn = 2 * TP + FP + FN
        return dict(dice=float(2 * TP / dn) if dn else 1.0, iou=float(TP / (TP + FP + FN)) if (TP + FP + FN) else 1.0,
                    precision=float(TP / (TP + FP)) if (TP + FP) else (1.0 if FN == 0 else 0.0),
                    recall=float(TP / (TP + FN)) if (TP + FN) else 1.0,
                    specificity=float(TN / (TN + FP)) if (TN + FP) else 1.0)
    allsel = np.ones(N, bool); G = pooled(allsel)
    summary = dict(
        threshold=thr, n_images=int(N), resolution=f"{SIZE}x{SIZE}",
        dice_per_image_mean=float(s["dice"].mean()), dice_per_image_std=float(s["dice"].std()), dice_per_image_median=float(np.median(s["dice"])),
        dice_global_pooled=G["dice"],
        iou_per_image_mean=float(s["iou"].mean()), iou_global_pooled=G["iou"],
        precision_per_image_mean=float(s["precision"].mean()), precision_global_pooled=G["precision"],
        recall_per_image_mean=float(s["recall"].mean()), recall_global_pooled=G["recall"],
        specificity_per_image_mean=float(s["specificity"].mean()), specificity_global_pooled=G["specificity"],
        hd95_px_mean_defined=float(hd_def.mean()) if len(hd_def) else float("nan"),
        hd95_px_median_defined=float(np.median(hd_def)) if len(hd_def) else float("nan"),
        hd95_px_mean_penalised=float(pen.mean()), hd95_penalty_px=HD_PENALTY,
        n_hd95_defined=int(len(hd_def)), n_hd95_undefined=int(np.isnan(hd).sum()),
        n_empty_pred=int((s["pred_px"] == 0).sum()), n_empty_gt=int((s["gt_px"] == 0).sum()), n_dice_zero=int((s["dice"] == 0).sum()))
    rowsc = []
    for c, name in enumerate(classes):
        sel = (np.asarray(y) == c); pc = pooled(sel) if sel.any() else dict(dice=np.nan, iou=np.nan)
        hc = hd[sel]; hcd = hc[~np.isnan(hc)]
        rowsc.append(dict(cls=name, n=int(sel.sum()),
                          dice_per_image_mean=float(s["dice"][sel].mean()) if sel.any() else np.nan,
                          dice_per_image_std=float(s["dice"][sel].std()) if sel.any() else np.nan,
                          dice_global_pooled=pc["dice"], iou_per_image_mean=float(s["iou"][sel].mean()) if sel.any() else np.nan,
                          hd95_px_mean_defined=float(hcd.mean()) if len(hcd) else np.nan, n_hd95_undefined=int(np.isnan(hc).sum()),
                          n_empty_pred=int((s["pred_px"][sel] == 0).sum())))
    per_img = pd.DataFrame(dict(dice=s["dice"], iou=s["iou"], precision=s["precision"], recall=s["recall"],
                                specificity=s["specificity"], hd95_px=hd, pred_px=s["pred_px"].astype(int), gt_px=s["gt_px"].astype(int)))
    return summary, pd.DataFrame(rowsc), per_img

In [ ]:
# ---------------- prediction / validation evaluation (torch) ----------------
@torch.no_grad()
def predict(model, dl, keep_masks=False, with_dice=True, mask_acc=None):
    """Returns class probabilities (N,NC), per-image Dice (or None) and, if keep_masks, mask probabilities (N,H,W) float32.
    If `mask_acc` (float32 array (N,H,W)) is given, each batch's mask probabilities are ADDED into it in place (ensemble inference without
    building a separate (N,512,512) array per model). Order == the order of the indices the loader was built from (shuffle=False)."""
    model.eval(); P, Ds, MP = [], [], []; off = 0
    for x, m, y, has in dl:
        x = x.to(dev, non_blocking=True).expand(-1, 3, -1, -1)
        with torch.autocast(dev.type, enabled=dev.type == "cuda"):
            out = model(x)
        assert out["seg"].shape[-2:] == x.shape[-2:] and out["cls"].shape[-1] == NC
        P.append(F.softmax(out["cls"].float(), 1).cpu())
        sm = torch.sigmoid(out["seg"].float())[:, 0].cpu(); b = sm.shape[0]
        if with_dice: Ds.append(dice_imgs(sm.numpy(), m[:, 0].numpy(), has.numpy()))
        if mask_acc is not None: mask_acc[off:off + b] += sm.numpy()
        if keep_masks: MP.append(sm)
        off += b; del out, sm
    P = torch.cat(P).numpy()
    return P, (np.concatenate(Ds) if with_dice else None), (torch.cat(MP).numpy() if keep_masks else None)

def evaluate(model, idx, keep_masks=False):
    """Used on VALIDATION folds only (early stopping / checkpoint selection)."""
    P, Ds, MP = predict(model, loader(idx, False, eval_bs()), keep_masks)
    r = cls_metrics(Y[idx], P); r["dice"] = float(Ds.mean()) if len(Ds) else float("nan")
    return r, P, MP

In [ ]:
# ---------------- one fold (training logic unchanged; the 16-image batch is processed in micro-batches with gradient accumulation) ----------------
def train_fold(fold, tr_idx, va_idx):
    seed_all(HP["seed"] + fold)
    model = FinalNet(NC).to(dev); ema = EMA(model, HP["ema"]); unc = Uncertainty().to(dev)
    cw = None
    if CFG["class_weights"]:
        c = np.bincount(Y[tr_idx], minlength=NC); cw = torch.tensor(len(tr_idx) / (NC * np.maximum(c, 1)), dtype=torch.float32, device=dev)
    ce = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)
    opt = torch.optim.AdamW([dict(params=model.parameters(), weight_decay=HP["wd"]),
                             dict(params=unc.parameters(), weight_decay=0.0, lr=1e-3)], lr=HP["lr"])
    dl = loader(tr_idx, True); steps = HP["epochs"] * len(dl); warm = len(dl)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
                                              else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")
    best, bad, path = -1, 0, f"{RES}/checkpoints/fold{fold}_best_ema.pt"
    hist, best_ep = [], -1
    if dev.type == "cuda": torch.cuda.reset_peak_memory_stats()
    for ep in range(HP["epochs"]):
        model.train(); t0 = time.time(); tl = 0
        for x, m, y, has in dl:                                   # dl yields the full effective batch (16, drop_last) exactly as in the baseline
            x = x.to(dev, non_blocking=True).expand(-1, 3, -1, -1); m = m.to(dev); y = y.to(dev); has = has.to(dev)
            n = x.shape[0]; opt.zero_grad(set_to_none=True); bl = 0.0
            mb = HP["micro_batch"]
            for xs, ms, ys, hs in zip(x.split(mb), m.split(mb), y.split(mb), has.split(mb)):
                with torch.autocast(dev.type, enabled=dev.type == "cuda"):
                    out = model(xs)
                ls = seg_loss(out, ms, hs); lc = ce(out["cls"].float(), ys)
                loss = unc(ls, lc) * (xs.shape[0] / n)            # weight by micro-batch share -> accumulated gradient = mean over the 16 images
                scaler.scale(loss).backward(); bl += loss.item(); del out, ls, lc, loss
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)     # clipping once per optimiser step, after accumulation
            scaler.step(opt); scaler.update(); sched.step(); ema.update(model); tl += bl
        r, _, _ = evaluate(ema.m, va_idx)                       # VALIDATION fold only - the test set is never touched here
        score = r["f1"] + (0 if math.isnan(r["dice"]) else r["dice"])
        print(f"fold{fold} ep{ep+1:02d} loss {tl/len(dl):.4f} | val acc {r['acc']:.4f} f1 {r['f1']:.4f} "
              f"dice {r['dice']:.4f} | w=({unc.s[0].item():.2f},{unc.s[1].item():.2f}) | {time.time()-t0:.0f}s")
        hist.append(dict(fold=fold, epoch=ep + 1, train_loss=tl / len(dl), val_acc=r["acc"], val_prec=r["prec"], val_rec=r["rec"],
                         val_f1=r["f1"], val_dice=r["dice"], val_score=score, lr=opt.param_groups[0]["lr"],
                         w_seg=unc.s[0].item(), w_cls=unc.s[1].item(), epoch_time_s=time.time() - t0))
        if score > best: best, bad, best_ep = score, 0, ep + 1; torch.save(ema.m.state_dict(), path)
        else:
            bad += 1
            if bad >= HP["patience"]: print("early stop"); break
    if dev.type == "cuda": print(f"fold{fold} peak CUDA memory: {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
    return path, hist, dict(best_epoch=best_ep, best_val_score=best, effective_batch=HP["batch"], micro_batch=HP["micro_batch"], accumulation=HP["accum"])

In [ ]:
# ---------------- run: K folds (validation used for early stopping / checkpoint selection ONLY; no test evaluation here) ----------------
print(f"train+val={len(tv_idx)} imgs / {len(P_tv)} patients | test={len(te_idx)} imgs / {len(P_te)} patients (untouched until the ensemble cell)")
paths, histories, fold_rows = [], [], []
for k, (tr, va) in enumerate(FOLDS):
    path, hist, info = train_fold(k, tr, va); paths.append(path); histories += hist
    mdl = FinalNet(NC, pretrained=False).to(dev); mdl.load_state_dict(torch.load(path, map_location=dev))
    r, _, _ = evaluate(mdl, va)                                   # best EMA checkpoint on its own VALIDATION fold
    fold_rows.append(dict(fold=k, n_train_images=len(tr), n_val_images=len(va), n_val_patients=len(ps(va)), **info,
                          **{f"val_{a}": b for a, b in r.items()}))
    print(f"==> fold {k} VAL (best EMA ckpt, epoch {info['best_epoch']}): " + "  ".join(f"{n} {v:.4f}" for n, v in r.items()))
    del mdl; torch.cuda.empty_cache()
    pd.DataFrame(histories).to_csv(f"{RES}/training_history.csv", index=False)      # saved after every fold (crash-safe)

HIST = pd.DataFrame(histories); FOLD_VAL = pd.DataFrame(fold_rows)
HIST.to_csv(f"{RES}/training_history.csv", index=False); HIST.to_json(f"{RES}/training_history.json", orient="records", indent=1)
FOLD_VAL.to_csv(f"{RES}/fold_validation_metrics.csv", index=False)
print(f"\n===== per-fold VALIDATION metrics (best EMA checkpoint; mean +- std over {len(paths)} folds) =====")
for key in ["val_acc", "val_prec", "val_rec", "val_f1", "val_dice"]:
    print(f"{key:9s} {np.nanmean(FOLD_VAL[key]):.4f} +- {np.nanstd(FOLD_VAL[key]):.4f}")
print("(validation folds were used for checkpoint selection, so these are optimistic; the unbiased estimate is the test-set ensemble below)")

# final ensemble checkpoint = the five best-EMA fold state-dicts in a single file (+ metadata)
torch.save(dict(state_dicts=[torch.load(p, map_location="cpu") for p in paths], classes=CLASSES, size=SIZE, seg_threshold=SEG_THR,
                hp=HP, seed=HP["seed"], arch_cfg=ARCH, split_source=SPLIT_SOURCE,
                arch="FinalNet-ASPP-FPN-512: EfficientNet-B0 + ASPP + FPN fusion + U-Net(scSE) + 3 aux heads + soft-mask fusion classifier",
                ensemble="average softmax probabilities and sigmoid mask probabilities over folds, then threshold"),
           f"{RES}/checkpoints/final_ensemble.pt")
print("saved:", [os.path.basename(p) for p in paths], "+ final_ensemble.pt")

In [ ]:
# ---------------- 5-fold ENSEMBLE: evaluated ONCE on the untouched hold-out test set ----------------
# One pass: average class probabilities and mask probabilities over the 5 fold models, THEN threshold at SEG_THR.
te_dl = loader(te_idx, False, eval_bs())
P_sum = 0; M_sum = np.zeros((len(te_idx), SIZE, SIZE), np.float32)      # mask probabilities are accumulated in place (no per-model copy)
for path in paths:
    mdl = FinalNet(NC, pretrained=False).to(dev); mdl.load_state_dict(torch.load(path, map_location=dev))
    P, _, _ = predict(mdl, te_dl, keep_masks=False, with_dice=False, mask_acc=M_sum)
    P_sum = P_sum + P; del mdl; torch.cuda.empty_cache()
P_ens = P_sum / len(paths); M_ens = M_sum; M_ens /= len(paths)             # average of the 5 sigmoid maps, THEN threshold at SEG_THR
del P_sum
Y_te, M_te = Y[te_idx], M[te_idx]
assert P_ens.shape == (len(te_idx), NC) and np.allclose(P_ens.sum(1), 1, atol=1e-4)
assert M_ens.shape == (len(te_idx), SIZE, SIZE) == M_te.shape

# keep the raw ensemble outputs so every number below can be recomputed without touching the test set again
np.savez_compressed(f"{RES}/test_ensemble_outputs.npz", test_index=te_idx, pid=PID[te_idx], y=Y_te, class_prob=P_ens.astype(np.float32),
                    mask_prob=M_ens.astype(np.float16), threshold=SEG_THR, size=SIZE)

# ---------------- Table A: classification ----------------
cls_overall, cls_per_class, CM = full_cls_report(Y_te, P_ens)
tableA = pd.DataFrame([dict(metric=k, value=v) for k, v in [("Accuracy", cls_overall["accuracy"]),
        ("Macro Precision", cls_overall["macro_precision"]), ("Macro Recall", cls_overall["macro_recall"]),
        ("Macro F1", cls_overall["macro_f1"]), ("Macro ROC-AUC (OvR)", cls_overall["macro_auc_ovr"])]])
print(f"===== TABLE A - classification, 5-fold ensemble on the hold-out test set ({len(te_idx)} images / {len(P_te)} patients) =====")
print(tableA.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\nPer-class (one-vs-rest):"); print(cls_per_class.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\nConfusion matrix (rows=true, cols=pred):", CLASSES); print(CM)

# ---------------- Table B: segmentation ----------------
seg_sum, seg_per_class, seg_img = full_seg_report(M_ens, M_te, Y_te)
tableB = pd.DataFrame([dict(metric=k, value=v) for k, v in [
    ("Dice (mean per-image, PRIMARY)", seg_sum["dice_per_image_mean"]), ("Dice (global, pooled pixels)", seg_sum["dice_global_pooled"]),
    ("IoU (mean per-image)", seg_sum["iou_per_image_mean"]), ("IoU (global)", seg_sum["iou_global_pooled"]),
    ("Pixel Precision (mean per-image)", seg_sum["precision_per_image_mean"]), ("Pixel Precision (global)", seg_sum["precision_global_pooled"]),
    ("Pixel Recall/Sensitivity (mean per-image)", seg_sum["recall_per_image_mean"]), ("Pixel Recall/Sensitivity (global)", seg_sum["recall_global_pooled"]),
    ("Specificity (mean per-image)", seg_sum["specificity_per_image_mean"]), ("Specificity (global)", seg_sum["specificity_global_pooled"]),
    ("HD95 px (mean, images where defined)", seg_sum["hd95_px_mean_defined"]), ("HD95 px (median, defined)", seg_sum["hd95_px_median_defined"]),
    (f"HD95 px (mean, undefined cases = {HD_PENALTY:.1f}px penalty)", seg_sum["hd95_px_mean_penalised"])]])
print(f"\n===== TABLE B - tumour segmentation, same ensemble & test set, threshold {SEG_THR}, {SIZE}x{SIZE} =====")
print(tableB.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\nempty predictions: {seg_sum['n_empty_pred']} | empty GT masks: {seg_sum['n_empty_gt']} | Dice==0 images: {seg_sum['n_dice_zero']} | "
      f"HD95 defined/undefined: {seg_sum['n_hd95_defined']}/{seg_sum['n_hd95_undefined']}  (none of these images were dropped from the means above, except undefined HD95 in the 'defined' variant)")
print("\nPer-class Dice (test images grouped by GROUND-TRUTH class):"); print(seg_per_class.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\nDice definition: PRIMARY = mean of per-image Dice over ALL test images (macro over images, every test image included); "
      "'global' = Dice of pooled pixels over the whole test set.")

# ---------------- save tables / json / csv ----------------
per_class_all = cls_per_class.merge(seg_per_class.rename(columns={"n": "n_images_seg"}), on="cls")
tableA.to_csv(f"{RES}/classification_results.csv", index=False)
tableB.to_csv(f"{RES}/segmentation_results.csv", index=False)
per_class_all.to_csv(f"{RES}/per_class_metrics.csv", index=False)
pimg = pd.DataFrame(dict(sample_index=te_idx, file=FILES[te_idx], pid=PID[te_idx], y_true=Y_te, y_pred=P_ens.argmax(1),
                         **{f"p_{c}": P_ens[:, i] for i, c in enumerate(CLASSES)})).join(seg_img)
pimg.to_csv(f"{RES}/test_per_image_results.csv", index=False)
common = dict(dataset=DATASET, seed=HP["seed"], n_test_images=int(len(te_idx)), n_test_patients=len(P_te), ensemble_members=len(paths),
              test_evaluated="once, after probability averaging over the five best-EMA fold models")
json.dump(jsonable(dict(**common, overall=cls_overall, per_class=cls_per_class.to_dict("records"),
                        confusion_matrix=dict(labels=CLASSES, rows_true_cols_pred=CM.tolist()),
                        fold_validation_mean_std={k: [float(np.nanmean(FOLD_VAL[k])), float(np.nanstd(FOLD_VAL[k]))] for k in ["val_acc", "val_prec", "val_rec", "val_f1", "val_dice"]})),
          open(f"{RES}/classification_metrics.json", "w"), indent=2)
json.dump(jsonable(dict(**common, overall=seg_sum, per_class=seg_per_class.to_dict("records"),
                        conventions="Dice primary = mean per-image Dice over all test images; empty/empty -> 1; HD95 in pixels at "
                                    f"{SIZE}x{SIZE} (NOT directly comparable with the 256x256 baseline; no physical spacing available -> no mm); undefined HD95 (exactly one mask empty) reported separately and as {HD_PENALTY:.1f}px-penalised mean")),
          open(f"{RES}/segmentation_metrics.json", "w"), indent=2)
json.dump(jsonable(dict(hp=HP, arch=ARCH, split_source=SPLIT_SOURCE, size=SIZE, seg_threshold=SEG_THR, classes=CLASSES, label_map=CFG["label_map"], seed=HP["seed"],
                        split_random_states=dict(holdout=int(ho_seed), cv=int(cv_seed)), torch=torch.__version__, smp=smp.__version__,
                        sklearn=sklearn.__version__, scipy=scipy.__version__, cuda=dev.type == "cuda")),
          open(f"{RES}/run_config.json", "w"), indent=2)
print("\nsaved tables / json / csv to", RES)

In [ ]:
# ---------------- figures: confusion matrix, ROC, per-class Dice, training curves, representative predictions ----------------
def plot_confusion(cm, classes, path):
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(5.2, 4.4)); im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, f"{cm[i, j]}\n({cmn[i, j]:.1%})", ha="center", va="center", fontsize=9, color="white" if cmn[i, j] > .5 else "black")
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=20); ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Confusion matrix - hold-out test, 5-fold ensemble")
    plt.colorbar(im, fraction=.046); plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

def plot_roc(y, P, classes, path):
    fig, ax = plt.subplots(figsize=(5.6, 5)); grid = np.linspace(0, 1, 201); tprs = []
    for c, name in enumerate(classes):
        yb = (y == c)
        if 0 < yb.sum() < len(yb):                                      # classes absent from the split are skipped safely
            fpr, tpr, _ = roc_curve(yb, P[:, c]); a = auc(fpr, tpr); ax.plot(fpr, tpr, label=f"{name} (AUC {a:.3f})")
            tprs.append(np.interp(grid, fpr, tpr))
        else: print(f"ROC: class '{name}' skipped (absent or only class in this split)")
    if tprs:
        mt = np.mean(tprs, 0); ax.plot(grid, mt, "k--", lw=2, label=f"macro-average (AUC {auc(grid, mt):.3f})")
    ax.plot([0, 1], [0, 1], ":", color="gray"); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title("One-vs-rest ROC - hold-out test, 5-fold ensemble"); ax.legend(loc="lower right"); plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

def plot_class_dice(seg_pc, path):
    fig, ax = plt.subplots(figsize=(5.2, 4)); x = np.arange(len(seg_pc))
    ax.bar(x - .18, seg_pc["dice_per_image_mean"], .36, yerr=seg_pc["dice_per_image_std"], capsize=3, label="mean per-image Dice (+-SD)")
    ax.bar(x + .18, seg_pc["dice_global_pooled"], .36, label="global pooled Dice")
    ax.set_xticks(x); ax.set_xticklabels([f"{c}\n(n={n})" for c, n in zip(seg_pc["cls"], seg_pc["n"])]); ax.set_ylim(0, 1.05)
    ax.set_ylabel("Dice"); ax.set_title("Per-class Dice - hold-out test"); ax.legend(loc="lower right", fontsize=8)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

def plot_history(hist, path):
    fig, ax = plt.subplots(1, 4, figsize=(17, 3.6))
    for f, h in hist.groupby("fold"):
        for a, col, t in zip(ax, ["train_loss", "val_acc", "val_f1", "val_dice"], ["train loss", "val accuracy", "val macro-F1", "val Dice (per-image)"]):
            a.plot(h["epoch"], h[col], label=f"fold {f}"); a.set_title(t); a.set_xlabel("epoch")
    ax[0].legend(fontsize=7); plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

def plot_representative(Xte, Mte, Pte, y, dice, classes, path, thr=SEG_THR):
    """Per class: worst / median / best test image by per-image Dice (qualitative only; picked after evaluation)."""
    picks = []
    for c in range(len(classes)):
        ix = np.where(y == c)[0]
        if len(ix):
            o = ix[np.argsort(dice[ix])]; picks += [(c, "worst", o[0]), (c, "median", o[len(o) // 2]), (c, "best", o[-1])]
    fig, ax = plt.subplots(len(picks), 4, figsize=(10, 2.5 * len(picks)), squeeze=False)
    for r, (c, tag, i) in enumerate(picks):
        gt, pr = Mte[i] > 0.5, Pte[i] > thr
        ax[r, 0].imshow(Xte[i], cmap="gray"); ax[r, 0].set_title(f"{classes[c]} - {tag}", fontsize=8)
        ax[r, 1].imshow(gt, cmap="gray"); ax[r, 1].set_title("ground truth", fontsize=8)
        ax[r, 2].imshow(pr, cmap="gray"); ax[r, 2].set_title(f"ensemble pred (Dice {dice[i]:.3f})", fontsize=8)
        ax[r, 3].imshow(Xte[i], cmap="gray"); ax[r, 3].set_title("overlay: GT green / pred red", fontsize=8)
        if gt.any(): ax[r, 3].contour(gt, levels=[.5], colors="lime", linewidths=1)
        if pr.any(): ax[r, 3].contour(pr, levels=[.5], colors="red", linewidths=1)
        for a in ax[r]: a.axis("off")
    plt.tight_layout(); plt.savefig(path, dpi=130); plt.show()

FIG = f"{RES}/figures"
plot_confusion(CM, CLASSES, f"{FIG}/confusion_matrix_test.png")
plot_roc(Y_te, P_ens, CLASSES, f"{FIG}/roc_curves_test.png")
plot_class_dice(seg_per_class, f"{FIG}/per_class_dice_test.png")
plot_history(HIST, f"{FIG}/training_history.png")
plot_representative(X[te_idx], M_te, M_ens, Y_te, seg_img["dice"].values, CLASSES, f"{FIG}/representative_segmentations.png")
print("figures saved to", FIG)

In [ ]:
# ---------------- paper comparison (only with VERIFIED published Figshare numbers) ----------------
# The original notebook compared with the paper's Table 13 for datasets D1/D2/D3 only. No published Figshare result with a verifiable
# protocol is available to us, so NO paper number is used or carried over from another dataset. To enable a comparison, fill PAPER_REF
# with numbers you have verified in the source, including how that source split the data.
PAPER_REF = None
# PAPER_REF = dict(source="<citation / table>", split_protocol="<how the paper split train/test (image-level? patient-level?)>",
#                  acc=None, f1=None, dice=None)
if PAPER_REF is None:
    print("Paper comparison skipped: no verified published Figshare results/protocol supplied (nothing is fabricated or reused from D1-D3).")
else:
    cmp = pd.DataFrame([dict(row="Original paper (as published)", acc=PAPER_REF["acc"], f1=PAPER_REF["f1"], dice=PAPER_REF["dice"]),
                        dict(row="Ours: 5-fold ensemble, patient-level hold-out", acc=cls_overall["accuracy"], f1=cls_overall["macro_f1"],
                             dice=seg_sum["dice_per_image_mean"])])
    print(cmp.to_string(index=False)); print("\nPaper source:", PAPER_REF["source"]); print("Paper split protocol:", PAPER_REF["split_protocol"])
    print("Ours: patient-level (PID) hold-out ~20% + patient-level 5-fold CV, test used once, 512x512, Dice = mean per-image over all test images.")
    print("Differences in splitting / resolution / Dice definition mean the two rows are NOT directly comparable unless the protocols match.")

# ---------------- comparison with the 256x256 FinalNet baseline (baseline numbers = the values reported from the previous run) ----------------
# Same hold-out test patients, same metric definitions, same 0.5 threshold. Read-only: nothing here feeds back into training or model selection.
BASE = dict(acc=0.9777, f1=0.9747, auc=0.9990, dice_mean=0.7774, dice_global=0.8132, iou_mean=0.6848, hd95_def=10.235,
            dice_men=0.9139, dice_gli=0.6782, dice_pit=0.8299, empty=17, zero=26)
pc = seg_per_class.set_index("cls")["dice_per_image_mean"]
cmp_rows = [("Accuracy", BASE["acc"], cls_overall["accuracy"]), ("Macro F1", BASE["f1"], cls_overall["macro_f1"]),
            ("Macro ROC-AUC (OvR)", BASE["auc"], cls_overall["macro_auc_ovr"]),
            ("Dice mean per-image (PRIMARY)", BASE["dice_mean"], seg_sum["dice_per_image_mean"]), ("Dice global pooled", BASE["dice_global"], seg_sum["dice_global_pooled"]),
            ("IoU mean per-image", BASE["iou_mean"], seg_sum["iou_per_image_mean"]),
            ("Dice meningioma", BASE["dice_men"], pc["meningioma"]), ("Dice glioma", BASE["dice_gli"], pc["glioma"]), ("Dice pituitary", BASE["dice_pit"], pc["pituitary"]),
            ("Empty predictions (count)", BASE["empty"], seg_sum["n_empty_pred"]), ("Zero-Dice images (count)", BASE["zero"], seg_sum["n_dice_zero"]),
            ("HD95 px, defined cases  [256 px vs 512 px: NOT comparable]", BASE["hd95_def"], seg_sum["hd95_px_mean_defined"])]
CMP = pd.DataFrame(cmp_rows, columns=["metric", "baseline_FinalNet_256", "ASPP_FPN_512"]); CMP["difference"] = CMP["ASPP_FPN_512"] - CMP["baseline_FinalNet_256"]
print("\n===== FinalNet-ASPP-FPN-512 vs FinalNet-256 baseline (same hold-out test set; descriptive, single 5-fold-ensemble run each) =====")
print(CMP.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("Caveats: (i) one run per configuration - differences of a few thousandths are within run-to-run noise (GPU non-determinism, seeds); "
      "(ii) the change bundles ASPP + FPN + 512 px, so this cannot attribute gains to any single component; "
      "(iii) HD95 is in pixels of the respective resolution - do not read it as a like-for-like or millimetre value.")
CMP.to_csv(f"{RES}/comparison_vs_baseline_256.csv", index=False)

**Notes / conventions — FinalNet-ASPP-FPN-512**
- **What changed vs. the 256-px FinalNet:** (A) `ASPP` module on the stride-32 bottleneck (cell 7); (B) `FPNFusion` over the five encoder stages whose outputs are added to the decoder skips (cell 7); (C) `SIZE = 512` for images, masks, dataset, model, aux outputs, prediction, evaluation, ensemble and figures; (D) gradient accumulation (micro-batch × accumulation = effective batch 16) and memory-lean prediction/ensemble code; (E) split lock + 629/46 assertion; (F) offline-safe imports. Everything else is textually unchanged (losses, optimiser, schedule, EMA, clipping, early stopping, metrics, threshold).
- **Effective batch:** every optimiser step still sees the same 16 images (the loader yields batch 16 with `drop_last`; they are processed in micro-batches). The accumulated gradient is the micro-batch-share-weighted mean, so CE/BCE gradients equal the full-batch ones. Two unavoidable differences: the Tversky term uses pooled TP/FP/FN *per micro-batch* (then averaged) instead of over all 16 images, and BatchNorm statistics are computed per micro-batch (running statistics are also updated once per micro-batch). `micro_batch` is chosen by the probe in cell 8 (`AUTO_MICRO_BATCH`) or set in `HP`.
- **Splitting:** everything is grouped by `PID` (StratifiedGroupKFold). The hold-out test set must equal the baseline's 629 images / 46 patients (asserted; `split_info.json` of the baseline is reused if attached). The split source is saved in `split_info.json`.
- **Model selection:** checkpoint selection and early stopping use only the validation fold (macro-F1 + per-image Dice, as in the baseline). The hold-out test set is touched **once**, by the 5-fold ensemble, after averaging probabilities. The baseline comparison cell is read-only and not used to tune anything.
- **Segmentation:** one binary tumour mask, threshold 0.5 applied after averaging the five models' sigmoid probabilities. **Primary Dice = mean of per-image Dice over *all* test images** (empty/empty = 1, missed or false-alarm = 0). Global pooled Dice/IoU/precision/recall/specificity are reported alongside. **HD95 is in pixels at 512×512** (MedPy-style); there is no physical pixel spacing, so no millimetre claim is made and it is not directly comparable with the 256-px numbers. Undefined HD95 cases (exactly one mask empty) are counted and also included in a penalised mean (image diagonal ≈ 724 px at 512).
- **Per-class Dice** groups test images by their ground-truth tumour class. **Classification:** macro precision/recall/F1; one-vs-rest AUC per class, macro-AUC = mean of the per-class AUCs (cross-checked against scikit-learn).
- **Pre-processing:** per-slice min-max → uint8, then the original normalisation `(x/255−0.449)/0.226`; 512-px slices are not resampled at `SIZE=512`, 256-px slices are up-sampled bilinearly; masks always `INTER_NEAREST` and asserted binary.
- If you hit CUDA OOM, set `AUTO_MICRO_BATCH=True` (default) or lower `HP["micro_batch"]` (4 or 2; the effective batch stays 16). GPU runs are not bit-wise deterministic even with fixed seeds. No result in this notebook has been pre-computed or assumed.